# cfcbench: CPU conic solvers vs cuOpt on CardilloCxx contact problems

Regenerates all figures from the JSON lines in `results/` (no solver runs here). Run the sweep with `cfcbench.run` or `scripts/bwuni_gpu.sbatch` first; keep the raw JSON next to every figure.

In [ ]:
import sys, glob
from pathlib import Path
sys.path.insert(0, '..')
from cfcbench import plots

RESULTS = sorted(glob.glob('../results/*.jsonl'))
runs = plots.load(RESULTS)
print(len(runs), 'runs from', RESULTS)

In [ ]:
# One line per run, sorted by scene, size, solver and tolerance.
for d in sorted(runs, key=lambda d: (d.get('family') or '', d.get('nc', 0), d['solver'], d['settings']['tol_target'])):
    r = d.get('residuals') or {}
    print(f"{d.get('family')!s:8} {d['problem']:22} nc={d.get('nc', 0):6} {d['form']} {d['solver']:9} tol={d['settings']['tol_target']:.0e} "
          f"{d['status'][:16]:16} it={d.get('iterations', -1):4} t={d.get('t_total', float('nan')):8.3f}s rel={r.get('rel_max', float('nan')):.2e}")

In [ ]:
TARGET = 1e-4
plots.style()
out = Path('../figures')
note = 'Platform: ' + '; '.join(sorted({(r.get('platform') or {}).get('cpu', '') for r in runs} - {''}))
plots.work_precision(runs, out, note)
plots.time_vs_contacts(runs, out, TARGET, note)
plots.performance_profile(runs, out, TARGET, note)
plots.speedup(runs, out, TARGET, note)

In [ ]:
from IPython.display import Image, display
for name in ['work_precision', 'time_vs_contacts', 'performance_profile', 'speedup']:
    p = out / f'{name}.png'
    if p.exists():
        display(Image(str(p)))